[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_6_change_detection/assignments/image_classification.ipynb)

# Assignment 6A: UAS image classification

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

In this assignment we classify the July 2024 orthophoto of Lake Wheeler into nine land cover classes. We derive features from the orthophoto and the DSM (a spectral index, spatial filters, texture, and terrain derivatives), train a random forest on digitized training polygons, map the classes, and measure the accuracy against polygons the model has not seen. In the last part we segment the same scene with a pretrained foundation model, SAM 2, from a text prompt and compare the result with the random forest classes.

Keep in mind that an accuracy computed on the training polygons describes how well the model memorized them, not how well it maps the rest of the site. The difference between the two numbers is one of the results of this assignment.

This notebook runs the same steps as the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_6_change_detection/assignments/assignment_6a.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed. For Part 10, switch Colab to a GPU runtime before you start (*Runtime > Change runtime type > T4 GPU*); the rest of the notebook does not need one.

## Learning objectives

- Derive spectral, spatial, textural, and terrain features from UAS products
- Group rasters into imagery groups and segment an image into superpixels
- Train a random forest, map its predictions, and read the feature importances
- Assess a classification with a confusion matrix on held-out reference data
- Segment an image with a text prompt and a pretrained model and compare it with a trained classifier

## Outline

- Part 1: Environment setup
- Part 2: Import and region
- Part 3: Spectral index
- Part 4: Spatial filters
- Part 5: Texture
- Part 6: Terrain features
- Part 7: Imagery groups and segmentation
- Part 8: Training data and model
- Part 9: Prediction and feature importance
- Part 10: Accuracy assessment
- Part 11: Segmentation with a foundation model
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the install cells and install the Python packages named in them into the Python that GRASS uses.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba, together with scikit-learn, which the random forest tools use. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5 scikit-learn pandas matplotlib

Install the packages for Part 11: [segment-geospatial](https://samgeo.gishub.org/) with its SAM 2 extra, which brings PyTorch, and groundingdino-py, which finds the objects named in a text prompt. The `transformers` version is held below 5 because the Grounding DINO text model that segment-geospatial uses does not run with newer releases. This also takes a few minutes; you can skip it if you do not plan to run Part 11.

In [ ]:
# GRASS tools run with the conda Python (the first python3 on the PATH), not
# with the notebook kernel, so install into that interpreter and check the
# import there.
!python3 -m pip install -q "segment-geospatial[samgeo2]" groundingdino-py "transformers<5"
!MPLBACKEND=Agg python3 -c "import torch, samgeo, groundingdino; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())"

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5). We also import pandas and matplotlib for the tables and plots.

In [ ]:
import io
import os
import subprocess
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB)
- The training polygons, `classification_training.gpkg` (64 polygons in nine classes), and the color rules for the classes

The orthophoto and the DSM of the July 17, 2024 flight are GeoTIFFs on the course bucket; `r.import` reads only the part of them we ask for, so they are not downloaded as files.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
TRAINING_URL = f"{BUCKET}/grassdata/classification_training.gpkg"
COLORS_URL = f"{BUCKET}/grassdata/classification_colors.txt"
ORTHO_URL = f"{BUCKET}/uas-flight-data/Lake%20Wheeler%20-%20NCSU/071724/odm_orthophoto.tif"
DSM_URL = f"{BUCKET}/uas-flight-data/Lake%20Wheeler%20-%20NCSU/071724/dsm.tif"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o classification_training.gpkg {TRAINING_URL}
!curl -sSL -o classification_colors.txt {COLORS_URL}
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `classification` inside the Lake Wheeler project, start the session there, and create a `Tools` object.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "classification"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session, overwrite=True)

print(tools.g_proj(flags="p").text)

### Install the addons

Five addons are used: `r.learn.ml2` (the tools `r.learn.train` and `r.learn.predict`, an interface to scikit-learn), `i.superpixels.slic` and `i.segment.stats` (segmentation), `r.confusionmatrix` (accuracy assessment), and `i.sam2` (text prompted segmentation). Addons installed after the `Tools` object was created are called with `gs.run_command` in this notebook.

In [ ]:
# The conda build of GRASS keeps its shared libraries in $GISBASE/lib, which
# the addon build on Colab does not search. Addons that use grass.pygrass
# (r.learn.ml2, r.dem) load libgrass_gis while g.extension builds their manual
# page, so register that directory with the system linker first (Colab only).
grass_lib = Path(os.environ["GISBASE"]) / "lib"
if sys.platform.startswith("linux") and os.access("/etc/ld.so.conf.d", os.W_OK):
    Path("/etc/ld.so.conf.d/grass.conf").write_text(f"{grass_lib}\n")
    subprocess.run(["ldconfig"], check=True)

NPROCS = os.cpu_count() or 1
for addon in ["r.learn.ml2", "i.superpixels.slic", "i.segment.stats", "r.confusionmatrix", "i.sam2"]:
    gs.run_command("g.extension", extension=addon, quiet=True)
print(f"Addons installed; {NPROCS} CPU cores available")

### Helpers

`describe` prints the statistics of a raster in the current region. `show` draws a raster with an optional legend and vector overlay. `rgb` draws a three band composite. `full_region` and `farm_region` switch the computational region between the whole mosaic and a 150 m block around the farm buildings that we use for detail maps. `timed` runs a GRASS tool and prints how long it took.

In [ ]:
def describe(raster, label=None):
    """Print the range, mean, and spread of a raster in the current region."""
    s = tools.r_univar(map=raster, format="json").json
    print(f"{label or raster}: {s['n']:,} cells, min {s['min']:.3f}, max {s['max']:.3f}, mean {s['mean']:.3f}, stddev {s['stddev']:.3f}")


def show(raster, title=None, legend=None, width=600, height=565, flags="b", vector=None, at=(5, 45, 3, 6)):
    """Draw one raster in the current region with a legend."""
    m = gj.Map(width=width, height=height, use_region=True)
    m.d_rast(map=raster)
    if vector:
        m.d_vect(map=vector, color="yellow", fill_color="none", width=2)
    if legend is not None:
        m.d_legend(raster=raster, at=at, title=legend, flags=flags, bgcolor="white", fontsize=11)
    m.d_barscale(at=(55, 5), flags="n")
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()


def rgb(group_bands, title=None, width=600, height=565, vector=None):
    """Draw a true color composite of three bands."""
    m = gj.Map(width=width, height=height, use_region=True)
    m.d_rgb(red=group_bands[0], green=group_bands[1], blue=group_bands[2])
    if vector:
        m.d_vect(map=vector, color="yellow", fill_color="none", width=2)
    m.d_barscale(at=(55, 5), flags="n")
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()


def full_region():
    """Set the computational region to the whole mosaic at 0.2 m."""
    tools.g_region(n=219815, s=219070, w=636565, e=637355, res=0.2)


def farm_region(res=0.2):
    """Set the computational region to the 150 m block around the farm buildings."""
    tools.g_region(n=219450, s=219300, w=637030, e=637180, res=res)


def timed(tool, **kwargs):
    """Run a GRASS tool by name and print the elapsed time."""
    start = time.time()
    gs.run_command(tool, overwrite=True, **kwargs)
    print(f"{tool}: {time.time() - start:.0f} s")

---
## Part 2: Import and region

The orthophoto was processed in WebODM and is stored as a cloud optimized GeoTIFF in UTM zone 17N at 7 cm resolution with four bands: red, green, blue, and an alpha band that is zero outside the mosaic. At 7 cm the mosaic has 122 million cells, and every feature below would take several minutes to compute. We work at 0.2 m instead: the region covers the whole mosaic with 14.7 million cells, and `r.import` reads, reprojects, and resamples the orthophoto to that grid (`extent=region`, `resolution=region`). Cars, hay bales, and crop rows are still several cells wide at 0.2 m.

In [ ]:
tools.g_region(n=219815, s=219070, w=636565, e=637355, res=0.2, flags="p")
tools.r_import(input=f"/vsicurl/{ORTHO_URL}", output="ortho", extent="region", resolution="region", resample="nearest")
print(tools.g_list(type="raster", pattern="ortho*", mapset=".").text)

The alpha band becomes a mask, so that all following tools skip the cells outside the mosaic. The DSM of the same flight is imported with bilinear resampling, which suits a continuous surface.

In [ ]:
RED, GREEN, BLUE = "ortho.1", "ortho.2", "ortho.3"
tools.r_mapcalc(expression="ortho_mask = if(ortho.4 > 0, 1, null())")
tools.r_mask(raster="ortho_mask")
tools.r_import(input=f"/vsicurl/{DSM_URL}", output="dsm", extent="region", resolution="region", resample="bilinear")
tools.r_colors(map="dsm", color="elevation")
for raster in [RED, GREEN, BLUE, "dsm"]:
    describe(raster)

#### Map: the orthophoto and the DSM

In [ ]:
rgb([RED, GREEN, BLUE], title="Lake Wheeler, July 17, 2024")
tools.r_relief(input="dsm", output="dsm_relief", zscale=2)
m = gj.Map(width=600, height=565, use_region=True)
m.d_shade(shade="dsm_relief", color="dsm", brighten=20)
m.d_legend(raster="dsm", at=(5, 45, 3, 6), title="DSM (m)", flags="b", bgcolor="white", fontsize=11)
m.d_barscale(at=(55, 5), flags="n")
m.show()

---
## Part 3: Spectral index

The [vegetation indices lab](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_6_change_detection/assignments/lab_vegetation_indices.html) defines the indices that can be computed from an RGB orthophoto. We use VARI, the visible atmospherically resistant index, (G - R) / (G + R - B).

The bands are 8-bit integers. In map algebra an integer divided by an integer gives an integer, so the first expression below returns only whole numbers. The second converts the numerator with `float()`. Its range is still very wide, because the denominator is close to zero on dark and on blue cells. The third sets cells with a denominator of 25 or less (about 0.1 in reflectance units) to null and clamps the result to -1 to 1.

In [ ]:
tools.r_mapcalc(expression=f"vari_int = ({GREEN} - {RED}) / ({GREEN} + {RED} - {BLUE})")
describe("vari_int", "VARI, integer division")
tools.r_mapcalc(expression=f"vari_raw = float({GREEN} - {RED}) / ({GREEN} + {RED} - {BLUE})")
describe("vari_raw", "VARI, floating point")
tools.r_mapcalc(expression=f"vari = if({GREEN} + {RED} - {BLUE} <= 25, null(), max(-1.0, min(1.0, float({GREEN} - {RED}) / ({GREEN} + {RED} - {BLUE}))))")
tools.r_colors(map="vari", color="ndvi")
describe("vari", "VARI, masked and clamped")
nulls = tools.r_univar(map="vari", format="json").json["null_cells"] - tools.r_univar(map=RED, format="json").json["null_cells"]
print(f"cells set to null by the denominator rule: {nulls:,}")

#### Map: VARI

The cells removed by the denominator rule are drawn in magenta in the second map.

In [ ]:
show("vari", title="VARI", legend="VARI")
tools.r_mapcalc(expression=f"vari_excluded = if(isnull(vari) && !isnull({RED}), 1, null())")
tools.r_colors(map="vari_excluded", rules=io.StringIO("1 magenta\n"))
m = gj.Map(width=600, height=565, use_region=True)
m.d_rgb(red=RED, green=GREEN, blue=BLUE)
m.d_rast(map="vari_excluded")
m.d_barscale(at=(55, 5), flags="n")
m.show()

> **Think about it:** Which surfaces lose cells to the denominator rule? The classifier cannot predict a cell where one of its features is null, so for the model we fill those cells: `vari_filled` replaces null with 0 inside the mosaic.

In [ ]:
tools.r_mapcalc(expression=f"vari_filled = if(isnull(vari) && !isnull({RED}), 0.0, vari)")
tools.r_colors(map="vari_filled", color="ndvi");

---
## Part 4: Spatial filters

A cell value alone says nothing about its surroundings. Moving window filters add that context.

A low pass filter smooths: `r.neighbors` with `method=average` in a 9 by 9 window (1.8 m) replaces each cell of the red band by the mean of its neighborhood, which removes the row pattern inside the crop fields. A high pass filter responds to local variation: the standard deviation of the green band in a 5 by 5 window (1 m) is high along edges and inside rough canopies and low on uniform surfaces such as roofs and the road.

In [ ]:
timed("r.neighbors", input=RED, output="red_mean9", size=9, method="average", nprocs=NPROCS)
timed("r.neighbors", input=GREEN, output="green_stddev5", size=5, method="stddev", nprocs=NPROCS)
tools.r_colors(map="red_mean9", color="grey")
tools.r_colors(map="green_stddev5", color="viridis", flags="e")
describe("red_mean9")
describe("green_stddev5")

#### Map: the two filters around the farm buildings

The maps zoom to the 150 m block around the farm buildings; the computational region is set back to the full mosaic afterwards.

In [ ]:
farm_region()
rgb([RED, GREEN, BLUE], title="Orthophoto", width=450, height=450)
show("red_mean9", title="Red, 9 by 9 mean", width=450, height=450)
show("green_stddev5", title="Green, 5 by 5 standard deviation", legend="DN", width=450, height=450)
full_region()

Edge detection is a third kind of filter. `i.zc` marks zero crossings of the Laplacian of the smoothed image; the output cells are zero except on edges, where the value gives the edge orientation. We compute it for the map only and do not use it as a feature.

In [ ]:
timed("i.zc", input=GREEN, output="green_zc", orientations=4, threshold=0.5)
farm_region()
show("green_zc", title="Zero crossing edges", width=450, height=450)
full_region()

---
## Part 5: Texture

`r.texture` computes the texture measures of Haralick et al. (1973) from the grey level co-occurrence matrix (GLCM) of a moving window: how often pairs of grey levels occur next to each other. We compute three of them for the green band in a 5 by 5 window, averaged over the four directions:

- angular second moment (ASM): high where the window is uniform
- contrast: high where neighboring cells differ strongly
- correlation: high where grey levels change gradually and predictably

This is the slowest feature of the assignment (two to four minutes).

In [ ]:
timed("r.texture", input=GREEN, output="green_tex", method="asm,contrast,corr", size=5, nprocs=NPROCS)
print(tools.g_list(type="raster", pattern="green_tex*", mapset=".").text)
for raster in ["green_tex_ASM", "green_tex_Contr", "green_tex_Corr"]:
    tools.r_colors(map=raster, color="viridis", flags="e")
    describe(raster)

#### Map: the three texture measures around the farm buildings

In [ ]:
farm_region()
show("green_tex_ASM", title="Angular second moment", width=450, height=450)
show("green_tex_Contr", title="Contrast", width=450, height=450)
show("green_tex_Corr", title="Correlation", width=450, height=450)
full_region()

---
## Part 6: Terrain features

Features do not have to come from the image. The DSM of the same flight separates tall objects from flat ones with the same color: trees from grass, roofs from gravel. From the DSM we derive slope, profile and tangential curvature with `r.slope.aspect`, and landforms with `r.geomorphon` (a search radius of 15 cells, 3 m). The DSM itself is also a feature.

In [ ]:
timed("r.slope.aspect", elevation="dsm", slope="slope", pcurvature="pcurv", tcurvature="tcurv", nprocs=NPROCS)
timed("r.geomorphon", elevation="dsm", forms="geomorphon", search=15, skip=0, flat=1, dist=0)
describe("slope")
print(tools.r_category(map="geomorphon").text)

#### Map: slope and landforms around the farm buildings

In [ ]:
farm_region()
show("slope", title="Slope of the DSM", legend="degrees", width=450, height=450)
show("geomorphon", title="Geomorphons", legend="", flags="cb", width=450, height=450, at=(5, 50, 3, 6))
full_region()

> **Think about it:** The DSM stores elevation above the vertical datum, between 75 and 116 m on this site. What does a model learn from that value: something about the land cover, or something about where on the hill a class was digitized?

---
## Part 7: Imagery groups and segmentation

GRASS imagery tools take their input as an imagery group, a named list of rasters. We create two: `rgb_group` with the three bands, and `features` with the bands and the eleven derived rasters.

In [ ]:
FEATURES = [RED, GREEN, BLUE, "vari_filled", "red_mean9", "green_stddev5",
            "green_tex_ASM", "green_tex_Contr", "green_tex_Corr",
            "dsm", "slope", "pcurv", "tcurv", "geomorphon"]
tools.i_group(group="rgb_group", subgroup="rgb_group", input=[RED, GREEN, BLUE])
tools.i_group(group="features", subgroup="features", input=FEATURES)
print(tools.i_group(group="features", flags="l").text)

### Superpixels

A pixel based classifier labels each cell on its own. An object based classification first segments the image into groups of similar neighboring cells and then classifies the groups. `i.superpixels.slic` implements SLIC (simple linear iterative clustering), which grows compact segments from a regular grid of seeds. We run it on the farm block with about 2,000 superpixels of at least 50 cells, and `i.segment.stats` computes the mean and standard deviation of each band and the area and compactness of each segment into the attribute table of a vector map.

In [ ]:
farm_region()
timed("i.superpixels.slic", input="rgb_group", output="superpixels", num_pixels=2000, compactness=1, minsize=50)
n_segments = len(tools.r_stats(input="superpixels", flags="n").text.split())
print(f"{n_segments:,} superpixels")
timed("i.segment.stats", map="superpixels", rasters=[RED, GREEN, BLUE], raster_statistics="mean,stddev",
      area_measures="area,compact_circle", vectormap="segment_stats", processes=NPROCS)
table = pd.read_csv(io.StringIO(tools.v_db_select(map="segment_stats", separator="comma").text))
table.head()

#### Map: superpixel outlines on the orthophoto

In [ ]:
m = gj.Map(width=600, height=600, use_region=True)
m.d_rgb(red=RED, green=GREEN, blue=BLUE)
m.d_vect(map="segment_stats", color="yellow", fill_color="none", width=1)
m.d_barscale(at=(55, 5), flags="n")
m.show()
full_region()

The table has one row per segment. It could be joined to training labels and classified with any tabular model; the rest of the assignment stays with the pixel based approach.

---
## Part 8: Training data and model

### Training polygons

The training data are 64 polygons digitized on the orthophoto in nine classes. The attribute `class_id` holds the class number and `class` its name. To test the model on data it has not seen, we set every third polygon aside: polygons whose category number is divisible by 3 become the test set, the others the training set. Both are converted to rasters whose values are the class numbers and whose labels are the class names.

In [ ]:
tools.v_import(input=str(DATA_DIR / "classification_training.gpkg"), layer="training", output="training")
tools.v_extract(input="training", where="cat % 3 != 0", output="training_train")
tools.v_extract(input="training", where="cat % 3 = 0", output="training_test")
for part in ["train", "test"]:
    tools.v_to_rast(input=f"training_{part}", output=f"{part}_rast", use="attr",
                    attribute_column="class_id", label_column="class")

def class_counts(raster):
    text = tools.r_stats(input=raster, flags="cln", separator="|").text
    return pd.read_csv(io.StringIO(text), sep="|", names=["class_id", "class", "cells"]).set_index(["class_id", "class"])

polygons = pd.read_csv(io.StringIO(tools.v_db_select(map="training", columns="cat,class_id,class", separator="comma").text))
counts = class_counts("train_rast").rename(columns={"cells": "training cells"}).join(
    class_counts("test_rast").rename(columns={"cells": "test cells"}))
counts.insert(0, "polygons", polygons.groupby(["class_id", "class"]).size())
counts

#### Map: training (yellow) and test (cyan) polygons

In [ ]:
m = gj.Map(width=600, height=565, use_region=True)
m.d_rgb(red=RED, green=GREEN, blue=BLUE)
m.d_vect(map="training_train", color="yellow", fill_color="none", width=2)
m.d_vect(map="training_test", color="cyan", fill_color="none", width=2)
m.d_barscale(at=(55, 5), flags="n")
m.show()

> **Think about it:** The classes are very unequal in size: the three field polygons hold more than 85 percent of the cells, and the crop class has fewer than 100. What does that mean for an overall accuracy?

### Train two random forests

A random forest is a set of decision trees, each trained on a bootstrap sample of the training cells and on a random subset of the features at each split; the class is the majority vote of the trees. `r.learn.train` trains a scikit-learn model on the cells of the group under the training raster and saves it to a file. We train two models with 100 trees each: one on the three bands only and one on all 14 features. `category_maps` tells the tool that the geomorphon raster holds categories, not quantities. The `-f` flag computes permutation feature importances and writes them to a CSV file.

In [ ]:
timed("r.learn.train", group="rgb_group", training_map="train_rast", model_name="RandomForestClassifier",
      n_estimators=100, random_state=1, n_jobs=NPROCS, save_model=str(DATA_DIR / "rf_rgb.gz"))
timed("r.learn.train", group="features", training_map="train_rast", model_name="RandomForestClassifier",
      n_estimators=100, random_state=1, n_jobs=NPROCS, category_maps="geomorphon",
      save_model=str(DATA_DIR / "rf_features.gz"), flags="f", fimp_file=str(DATA_DIR / "feature_importance.csv"))

---
## Part 9: Prediction and feature importance

`r.learn.predict` loads a model and predicts the class of every cell of the group in the computational region. The class names travel with the model, and the color rules file gives each class a color.

In [ ]:
timed("r.learn.predict", group="rgb_group", load_model=str(DATA_DIR / "rf_rgb.gz"), output="rf_rgb")
timed("r.learn.predict", group="features", load_model=str(DATA_DIR / "rf_features.gz"), output="rf_features")
for raster in ["rf_rgb", "rf_features"]:
    tools.r_colors(map=raster, rules=str(DATA_DIR / "classification_colors.txt"))
print(tools.r_category(map="rf_features").text)

#### Map: the two classifications

In [ ]:
show("rf_rgb", title="Random forest, RGB bands", legend="", flags="cb", at=(5, 40, 3, 6))
show("rf_features", title="Random forest, 14 features", legend="", flags="cb", at=(5, 40, 3, 6))

In [ ]:
farm_region()
show("rf_rgb", title="RGB bands", width=450, height=450)
show("rf_features", title="14 features", width=450, height=450)
full_region()

### Class areas

The area each class covers in the two maps, from `r.stats`.

In [ ]:
def class_areas(raster):
    text = tools.r_stats(input=raster, flags="aln", separator="|").text
    t = pd.read_csv(io.StringIO(text), sep="|", names=["class_id", "class", "m2"]).set_index("class")
    return (t["m2"] / 10000).round(2)

areas = pd.DataFrame({"RGB bands (ha)": class_areas("rf_rgb"), "14 features (ha)": class_areas("rf_features")})
areas

### Feature importance

The permutation importance of a feature is the drop in accuracy when its values are shuffled among the training cells: a large drop means the model depends on that feature.

#### Plot: feature importance of the 14 feature model

In [ ]:
importance = pd.read_csv(DATA_DIR / "feature_importance.csv").sort_values("importance")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(importance["feature"], importance["importance"], xerr=importance["std"], color="#4c8c4a")
ax.set_xlabel("Permutation importance (drop in accuracy)")
ax.set_title("Random forest, 14 features")
fig.tight_layout()
plt.show()
importance.sort_values("importance", ascending=False).round(4)

---
## Part 10: Accuracy assessment

`r.confusionmatrix` cross tabulates a classification with a reference raster and derives the accuracy measures:

- overall accuracy: correctly classified cells divided by all reference cells
- user's accuracy of a class: of the cells mapped as the class, the share that is the class in the reference (100 percent minus the commission error)
- producer's accuracy of a class: of the reference cells of the class, the share that was mapped as the class (100 percent minus the omission error)
- kappa: the agreement beyond what chance would produce

We compute them twice for each model: against the training polygons, which the model has seen, and against the test polygons, which it has not.

In [ ]:
def confusion(classification, reference):
    """Run r.confusionmatrix and return (matrix as DataFrame, overall accuracy, kappa)."""
    out = DATA_DIR / f"confusion_{classification}_{reference}.csv"
    # read_command captures the matrix that the tool also prints
    gs.read_command("r.confusionmatrix", classification=classification, raster_reference=reference,
                    csvfile=str(out), flags="m", overwrite=True, quiet=True)
    rows = [line.rstrip("\n").split(",") for line in open(out)]
    header = rows[1][2:]
    body = [r[1:] for r in rows[2:]]
    matrix = pd.DataFrame([r[1:] for r in body], index=[r[0] for r in body], columns=header)
    overall = float(matrix.loc["Producer Accuracy"].iloc[-1])
    kappa = float(matrix.loc["Omission Error"].iloc[-1])
    return matrix, overall, kappa

summary = []
matrices = {}
for model in ["rf_rgb", "rf_features"]:
    for reference in ["train_rast", "test_rast"]:
        matrix, overall, kappa = confusion(model, reference)
        matrices[(model, reference)] = matrix
        summary.append({"model": model, "reference": reference, "overall accuracy (%)": overall, "kappa": kappa})
pd.DataFrame(summary)

#### Table: confusion matrix of the 14 feature model on the test polygons

Rows are the classes of the map, columns the classes of the reference.

In [ ]:
matrices[("rf_features", "test_rast")]

#### Table: confusion matrix of the RGB model on the test polygons

In [ ]:
matrices[("rf_rgb", "test_rast")]

> **Think about it:** Find the largest off-diagonal number in the first table. Which test polygon produces it, and what does the model see there that it also saw in the training polygons of the other class?

### One change to the model

Change one thing and repeat the training, prediction, and assessment on the test polygons. The cell below is one example: it replaces the DSM by the height above the lowest cell within 7.5 m, which describes the height of an object and not its position on the hill. Other options: remove a feature, balance the classes with the `-b` flag, use more trees, or another `model_name`.

In [ ]:
timed("r.neighbors", input="dsm", output="dsm_min", size=75, method="minimum", nprocs=NPROCS)
tools.r_mapcalc(expression="height = dsm - dsm_min")
tools.i_group(group="features_height", subgroup="features_height",
              input=[f if f != "dsm" else "height" for f in FEATURES])
timed("r.learn.train", group="features_height", training_map="train_rast", model_name="RandomForestClassifier",
      n_estimators=100, random_state=1, n_jobs=NPROCS, category_maps="geomorphon",
      save_model=str(DATA_DIR / "rf_height.gz"))
timed("r.learn.predict", group="features_height", load_model=str(DATA_DIR / "rf_height.gz"), output="rf_height")
tools.r_colors(map="rf_height", rules=str(DATA_DIR / "classification_colors.txt"))
matrix, overall, kappa = confusion("rf_height", "test_rast")
print(f"height in place of DSM: overall accuracy {overall} %, kappa {kappa}")
matrix

---
## Part 11: Segmentation with a foundation model

The random forest learned the classes from our polygons. A foundation model is trained once on a very large collection of images and then used without further training. `i.sam2` runs two of them through the segment-geospatial package: Grounding DINO finds boxes in the image that match a text prompt, and SAM 2 (Segment Anything Model 2) outlines the object in each box. The output raster is 0 in the background and numbers the objects from 1.

The tool reads the whole computational region as one image, so we use the farm block at 0.1 m (1,500 by 1,500 cells). The model weights (about 1.5 GB) are downloaded on the first run. On a Colab GPU a prompt takes a few seconds; on a CPU about a minute.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
print(f"i.sam2 will run on: {device}")

# The notebook sets MPLBACKEND to its inline backend, which the Python that
# runs GRASS tools does not have. Tools started from here get a file backend;
# plots in the notebook itself are not affected.
os.environ["MPLBACKEND"] = "Agg"

farm_region(res=0.1)
tools.r_import(input=f"/vsicurl/{ORTHO_URL}", output="farm_ortho", extent="region", resolution="region", resample="nearest")
tools.i_group(group="farm_rgb", subgroup="farm_rgb", input=["farm_ortho.1", "farm_ortho.2", "farm_ortho.3"]);

In [ ]:
for prompt in ["trees", "buildings", "cars"]:
    timed("i.sam2", group="farm_rgb", output=f"sam_{prompt}", text_prompt=prompt)
    objects = int(tools.r_info(map=f"sam_{prompt}", flags="r", format="json").json["max"])
    print(f"  prompt '{prompt}': {objects} objects")

#### Map: the objects found for each prompt

In [ ]:
rgb(["farm_ortho.1", "farm_ortho.2", "farm_ortho.3"], title="Orthophoto, 0.1 m", width=450, height=450)
for prompt in ["trees", "buildings", "cars"]:
    tools.r_mapcalc(expression=f"sam_{prompt}_objects = if(sam_{prompt} > 0, sam_{prompt}, null())")
    tools.r_colors(map=f"sam_{prompt}_objects", color="random")
    m = gj.Map(width=450, height=450, use_region=True)
    m.d_rgb(red="farm_ortho.1", green="farm_ortho.2", blue="farm_ortho.3")
    m.d_rast(map=f"sam_{prompt}_objects")
    m.d_text(text=f"prompt: {prompt}", at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()

### Thresholds

`box_threshold` is the minimum confidence of a box, `text_threshold` the minimum similarity between the box and the words of the prompt; both default to 0.24. Raising the box threshold keeps only the boxes the model is more certain about.

In [ ]:
for threshold in [0.24, 0.35, 0.5]:
    name = f"sam_buildings_{int(threshold * 100)}"
    try:
        gs.run_command("i.sam2", group="farm_rgb", output=name, text_prompt="buildings",
                       box_threshold=threshold, overwrite=True, quiet=True)
        objects = int(tools.r_info(map=name, flags="r", format="json").json["max"])
    except gs.CalledModuleError:
        objects = 0  # the tool stops with an error when no box passes the threshold
    print(f"box_threshold {threshold}: {objects} objects")

### Comparison with the random forest

For trees, buildings, and cars we compare the SAM mask with the cells of the same class in the 14 feature random forest map, inside the farm block: the area of each, the area of their intersection, and the intersection over union (IoU), which is 1 for identical masks and 0 for masks that do not overlap.

In [ ]:
rows = []
for prompt, class_id in [("trees", 2), ("buildings", 1), ("cars", 6)]:
    tools.r_mapcalc(expression=f"cmp_{prompt} = if(sam_{prompt} > 0, 1, 0) + if(rf_features == {class_id}, 2, 0)")
    tools.r_category(map=f"cmp_{prompt}", separator="pipe",
                     rules=io.StringIO("0|neither\n1|SAM only\n2|random forest only\n3|both\n"))
    tools.r_colors(map=f"cmp_{prompt}", rules=io.StringIO("0 white\n1 blue\n2 orange\n3 black\n"))
    text = tools.r_stats(input=f"cmp_{prompt}", flags="an", separator="|").text
    a = {int(line.split("|")[0]): float(line.split("|")[1]) for line in text.splitlines()}
    sam, rf, both = a.get(1, 0) + a.get(3, 0), a.get(2, 0) + a.get(3, 0), a.get(3, 0)
    rows.append({"prompt": prompt, "SAM (m2)": round(sam), "random forest (m2)": round(rf),
                 "both (m2)": round(both), "IoU": round(both / (sam + rf - both), 2)})
pd.DataFrame(rows)

#### Map: SAM only (blue), random forest only (orange), both (black)

In [ ]:
for prompt in ["trees", "buildings", "cars"]:
    show(f"cmp_{prompt}", title=prompt, legend="", flags="cb", width=450, height=450, at=(5, 30, 3, 6))
full_region()

> **Think about it:** The random forest needed training polygons and features but labels every cell of the site. SAM needed only a word but returns objects in one image tile. Which errors does each make in the farm block?

---
## Report questions

Answer in the cells below, or in your report document, as described on the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_6_change_detection/assignments/assignment_6a.html). Keep the answers short and refer to the outputs above.

#### Question 1: Region and features

Report the resolution and the number of cells of the computational region. Show the VARI map and one filter or texture map, and describe which surfaces each separates.

`Add text here with your answer to question 1`

#### Question 2: VARI outliers

Which surfaces were excluded by the denominator rule, and why do they produce extreme values? Name one other way to handle them.

`Add text here with your answer to question 2`

#### Question 3: Classification

Show the two classification maps. Where do the RGB model and the 14 feature model differ most, and which features explain it according to the importance plot?

`Add text here with your answer to question 3`

#### Question 4: Accuracy

Report the overall accuracy and kappa of both models on the training and on the test polygons, and the confusion matrix of the 14 feature model on the test polygons. Which classes are mapped well and which are confused? Why are the training numbers so much higher?

`Add text here with your answer to question 4`

#### Question 5: One change

Describe the change you made to the model and report the accuracy on the test polygons before and after. Did it help, and why?

`Add text here with your answer to question 5`

#### Question 6: SAM 2

Show the objects found for two prompts and the comparison table. How did the box threshold change the result? For which task on this site would you use the random forest and for which the text prompt?

`Add text here with your answer to question 6`